## Auto Loader — `cloudFiles`, the workhorse

Auto Loader is a Structured Streaming source identified by the format string `cloudFiles`. It does one job at any scale: **discover new files in a cloud directory and stream them into a Delta table**, with checkpointed exactly-once writes.

```python
(spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("cloudFiles.schemaLocation", schema_path)
   .load(source_path)
 .writeStream
   .option("checkpointLocation", checkpoint_path)
   .trigger(availableNow=True)
   .toTable("fintech_dev.bronze.card_transactions_raw"))
```

Key options the exam tests:

- **`cloudFiles.format`** — the underlying file format (`json`, `csv`, `parquet`, `avro`, `binaryFile`, `text`).
- **`cloudFiles.schemaLocation`** — where Auto Loader caches the inferred schema across runs.
- **`cloudFiles.schemaEvolutionMode`** — what to do when a new column appears.
- **`checkpointLocation`** (on the writer) — where processed offsets are tracked.
- **`trigger(availableNow=True)`** — process every file that exists now, then stop; the modern replacement for `trigger.once`.

**Auto Loader beats `COPY INTO` when:** files are huge in number, arrive continuously (not one daily drop), you want schema evolution baked in, or you may flip from scheduled to continuous later without a rewrite.